# Speaker-Independent Speech Emotion Recognition Across Three Corpora

This notebook presents the public analysis workflow for a four-class speech emotion recognition (SER) baseline across **RAVDESS**, **EMO-DB**, and **CREMA-D**. The target emotions are Neutral, Happy, Sad, and Angry.

The evaluation is speaker-independent: each test fold contains a speaker who is absent from the corresponding training fold. This prevents the model from benefiting from speaker identity leakage.


## Public-release note

Raw audio and full derived feature tables are not included in the repository because the source corpora have their own access and licensing conditions. To rerun the full evaluation, obtain each corpus from its official source, extract the same 31 acoustic features, and point `SER_DATA_ROOT` to the private feature directory.

The notebook contains no personal Google Drive path, volunteer recording, authentication token, or private identifier. Saved execution outputs from the working notebook were cleared before publication.


## Method summary

- Audio preprocessing: mono, 16 kHz, leading/trailing silence trimmed at 30 dB.
- Features: duration; mean and standard deviation of RMS energy and pitch; mean and standard deviation of 13 MFCCs (31 features total).
- Model: RBF-kernel support vector machine with class balancing.
- Preprocessing: median imputation for failed pitch estimates and standardization, both fitted inside each training fold.
- Validation: leave-one-speaker-out cross-validation within each corpus.
- Metrics: accuracy, macro F1, per-emotion recall, confusion matrix, and speaker-level macro F1.


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import librosa
except ImportError:
    librosa = None

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
EMOTION_ORDER = ["Neutral", "Happy", "Sad", "Angry"]


## Paths and run mode

The repository remains usable without the private feature tables: reported aggregate results are loaded from `results/` when available. To rerun the full speaker-independent evaluation, set the `SER_DATA_ROOT` environment variable and change `RUN_FULL_EVALUATION` to `True`.

Expected private layout:

```text
SER_DATA_ROOT/
├── RAVDESS/ser_features.csv
├── EMODB/emodb_features.csv
└── CREMA-D/crema_features.csv
```


In [ ]:
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

# Keep licensed or private corpus-derived files outside the public repository.
DATA_ROOT = Path(os.environ.get("SER_DATA_ROOT", REPO_ROOT / "private_data"))

FEATURE_FILES = {
    "RAVDESS": DATA_ROOT / "RAVDESS" / "ser_features.csv",
    "EMO-DB": DATA_ROOT / "EMODB" / "emodb_features.csv",
    "CREMA-D": DATA_ROOT / "CREMA-D" / "crema_features.csv",
}

GROUP_COLUMNS = {
    "RAVDESS": "actor_id",
    "EMO-DB": "speaker_id",
    "CREMA-D": "speaker_id",
}

RUN_FULL_EVALUATION = False
SAVE_REGENERATED_OUTPUTS = False

missing_files = [str(path) for path in FEATURE_FILES.values() if not path.exists()]
if missing_files:
    print("Private feature files are not configured. This is expected for the public repository.")
else:
    print("All private feature tables were found.")


## Acoustic feature extraction

This shared function was used to extract an identical feature set from all three corpora. Dataset-specific filename parsing is intentionally excluded from the public notebook; it depends on the official corpus distribution and metadata format.


In [ ]:
FEATURE_COLUMNS = [
    "duration",
    "energy_mean", "energy_std",
    "pitch_mean", "pitch_std",
] + [
    f"mfcc_{index}_{stat}"
    for index in range(1, 14)
    for stat in ("mean", "std")
]


def extract_acoustic_features(file_path, target_sr=16_000):
    # Extract the 31 acoustic features used in the baseline.
    if librosa is None:
        raise ImportError(
            "librosa is required for raw-audio feature extraction. "
            "Install the repository requirements before using this function."
        )

    signal, sample_rate = librosa.load(file_path, sr=target_sr, mono=True)
    signal, _ = librosa.effects.trim(signal, top_db=30)

    if len(signal) < 512:
        raise ValueError("Audio is empty or too short after trimming.")

    features = {"duration": len(signal) / sample_rate}

    rms = librosa.feature.rms(y=signal)[0]
    features["energy_mean"] = float(np.mean(rms))
    features["energy_std"] = float(np.std(rms))

    pitch, _, _ = librosa.pyin(
        signal,
        fmin=50,
        fmax=500,
        sr=sample_rate,
    )
    valid_pitch = pitch[np.isfinite(pitch)]
    if len(valid_pitch):
        features["pitch_mean"] = float(np.mean(valid_pitch))
        features["pitch_std"] = float(np.std(valid_pitch))
    else:
        features["pitch_mean"] = np.nan
        features["pitch_std"] = np.nan

    mfcc = librosa.feature.mfcc(y=signal, sr=sample_rate, n_mfcc=13)
    for index in range(13):
        features[f"mfcc_{index + 1}_mean"] = float(np.mean(mfcc[index]))
        features[f"mfcc_{index + 1}_std"] = float(np.std(mfcc[index]))

    return features


assert len(FEATURE_COLUMNS) == 31


## Speaker-independent evaluation

`SimpleImputer` and `StandardScaler` are inside the model pipeline. They are therefore fitted only on each training fold, rather than on the full corpus, which avoids preprocessing leakage.


In [ ]:
def evaluate_corpus(feature_table, group_column):
    required = set(FEATURE_COLUMNS + ["emotion", group_column])
    missing = sorted(required.difference(feature_table.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    data = feature_table.loc[
        feature_table["emotion"].isin(EMOTION_ORDER)
    ].copy()

    X = data[FEATURE_COLUMNS].copy()
    y = data["emotion"].copy()
    groups = data[group_column].astype(str)

    # Earlier extraction runs stored failed pitch values as zero.
    failed_pitch = X["pitch_mean"].eq(0)
    X.loc[failed_pitch, ["pitch_mean", "pitch_std"]] = np.nan

    splitter = LeaveOneGroupOut()
    predictions = np.empty(len(data), dtype=object)
    speaker_rows = []

    for train_index, test_index in splitter.split(X, y, groups):
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("svm", SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                class_weight="balanced",
            )),
        ])

        model.fit(X.iloc[train_index], y.iloc[train_index])
        fold_predictions = model.predict(X.iloc[test_index])
        predictions[test_index] = fold_predictions

        test_speaker = groups.iloc[test_index].iloc[0]
        speaker_rows.append({
            "speaker_id": test_speaker,
            "number_of_files": len(test_index),
            "accuracy": accuracy_score(y.iloc[test_index], fold_predictions),
            "macro_f1": f1_score(
                y.iloc[test_index],
                fold_predictions,
                labels=EMOTION_ORDER,
                average="macro",
                zero_division=0,
            ),
        })

    prediction_table = data[[group_column, "emotion"]].copy()
    if "filename" in data.columns:
        prediction_table.insert(0, "filename", data["filename"])
    prediction_table["predicted_emotion"] = predictions

    speaker_table = pd.DataFrame(speaker_rows)
    recalls = recall_score(
        y,
        predictions,
        labels=EMOTION_ORDER,
        average=None,
        zero_division=0,
    )
    matrix = confusion_matrix(
        y,
        predictions,
        labels=EMOTION_ORDER,
        normalize="true",
    )

    metrics = {
        "number_of_recordings": len(data),
        "number_of_speakers": groups.nunique(),
        "accuracy": accuracy_score(y, predictions),
        "macro_f1": f1_score(
            y,
            predictions,
            labels=EMOTION_ORDER,
            average="macro",
            zero_division=0,
        ),
        "mean_speaker_macro_f1": speaker_table["macro_f1"].mean(),
        "recall": dict(zip(EMOTION_ORDER, recalls)),
    }

    return metrics, prediction_table, speaker_table, matrix


In [ ]:
evaluation_outputs = {}

if RUN_FULL_EVALUATION:
    missing = [str(path) for path in FEATURE_FILES.values() if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Set SER_DATA_ROOT to a directory containing all three private feature tables. "
            f"Missing: {missing}"
        )

    for corpus, feature_path in FEATURE_FILES.items():
        feature_table = pd.read_csv(feature_path)
        evaluation_outputs[corpus] = evaluate_corpus(
            feature_table,
            GROUP_COLUMNS[corpus],
        )
        print(f"Completed speaker-independent evaluation: {corpus}")
else:
    print("Full evaluation skipped. Set RUN_FULL_EVALUATION = True after configuring private data.")


## Reported baseline results

These aggregate values come from the completed experiments documented in the project report. They are included so that the public notebook remains interpretable without redistributing the source corpora or full derived feature tables.


In [ ]:
reported_summary = pd.DataFrame({
    "Corpus": ["RAVDESS", "EMO-DB", "CREMA-D"],
    "Number_of_Recordings": [672, 339, 4898],
    "Number_of_Speakers": [24, 10, 91],
    "Accuracy_%": [66.22, 80.83, 69.44],
    "Overall_Macro_F1_%": [64.78, 81.69, 69.28],
    "Mean_Speaker_Macro_F1_%": [62.11, 78.51, 67.46],
    "Neutral_Recall_%": [59.38, 92.41, 71.73],
    "Happy_Recall_%": [63.54, 60.56, 57.44],
    "Sad_Recall_%": [61.46, 98.39, 71.65],
    "Angry_Recall_%": [77.08, 76.38, 77.26],
})

reported_summary


In [ ]:
reported_variability = pd.DataFrame({
    "Corpus": ["RAVDESS", "EMO-DB", "CREMA-D"],
    "Number_of_Speakers": [24, 10, 91],
    "Mean_Macro_F1_%": [62.11, 78.51, 67.46],
    "SD_%": [13.26, 10.39, 13.04],
    "Minimum_%": [39.22, 65.17, 25.77],
    "Median_%": [61.86, 75.18, 67.79],
    "Maximum_%": [84.29, 97.34, 92.65],
    "Range_%": [45.07, 32.17, 66.88],
})

reported_variability


## Comparison figures

The figures below are generated from the reported experiments and stored in the repository.

![Within-corpus performance comparison](../results/figures/three_corpus_performance_comparison.png)

![Per-emotion recall heatmap](../results/figures/three_corpus_recall_heatmap.png)

![Speaker-level performance variability](../results/figures/speaker_performance_variability.png)


## Interpretation

EMO-DB produced the strongest within-corpus results, while RAVDESS and CREMA-D were lower despite using the same feature set and model. The speaker-level distributions also show substantial variability, especially in CREMA-D. These results support treating corpus and speaker differences as important experimental factors rather than pooling recordings without controls.

This notebook reports within-corpus baselines only. Cross-corpus generalization and emotional-intensity mismatch are separate evaluation stages.
